# 0502 · 00 · Preparar el entorno

Laboratorio 0502 · Streaming tables y Auto Loader.

Ejecuta este notebook con **Run all** antes de empezar. Hace cuatro cosas:

1. Comprueba que los CSV de ventas están en la carpeta `datasets/sales-csv/` del repositorio y que cada uno tiene las filas esperadas.
2. Crea el catálogo `lab0502` con dos schemas:
   - `streaming`: donde se crean las tablas del laboratorio, con un volumen `checkpoints` para el estado de Auto Loader.
   - `upload`: con el volumen `upload`, la carpeta de aterrizaje de los ficheros.
3. Deja el laboratorio en su **estado inicial**: borra las tablas de una ejecución anterior, vacía los volúmenes y carga **solo `000.csv`**. El resto de ficheros los irás añadiendo tú durante la demo.
4. Verifica que el volumen contiene `000.csv` con **3.149 filas**.

Si quieres repetir el laboratorio desde cero, vuelve a ejecutarlo: siempre deja el mismo punto de partida.

In [ ]:
import os

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder;
# los datasets viven un nivel más arriba, en la raíz del repositorio.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_DATASETS = f"{os.path.dirname(RUTA_LAB)}/datasets/sales-csv"

CATALOGO = "lab0502"
VOLUMEN_UPLOAD = f"/Volumes/{CATALOGO}/upload/upload"
VOLUMEN_CHECKPOINTS = f"/Volumes/{CATALOGO}/streaming/checkpoints"

print("Laboratorio:", RUTA_LAB)
print("Datasets:   ", RUTA_DATASETS)
print("Catálogo:   ", CATALOGO)

## 1 · Comprobar los ficheros del repositorio

Los cuatro CSV de ventas van separados por `|`, con cabecera y con el JSON de `items` escapado con `\`. Se cuentan con un lector CSV real (no por líneas) para que el recuento sea fiable.

In [ ]:
import csv

FILAS_ESPERADAS = {"000.csv": 3149, "001.csv": 2932, "002.csv": 3200, "003.csv": 3203}

if not os.path.isdir(RUTA_DATASETS):
    raise FileNotFoundError(
        f"No encuentro {RUTA_DATASETS}. Comprueba que tu Git folder está actualizado "
        "y que la carpeta datasets/ está en la raíz del repositorio.")

errores = []
for fichero, esperadas in FILAS_ESPERADAS.items():
    ruta = f"{RUTA_DATASETS}/{fichero}"
    if not os.path.exists(ruta):
        errores.append(f"falta {fichero}")
        continue
    with open(ruta, newline="", encoding="utf-8") as f:
        lector = csv.reader(f, delimiter="|", quotechar='"', escapechar="\\", doublequote=False)
        filas = sum(1 for _ in lector) - 1  # sin la cabecera
    estado = "ok" if filas == esperadas else f"ESPERABA {esperadas}"
    print(f"{fichero}: {filas} filas  {estado}")
    if filas != esperadas:
        errores.append(f"{fichero} tiene {filas} filas")

if errores:
    raise ValueError("Ficheros incorrectos: " + "; ".join(errores))

## 2 · Catálogo, schemas y volúmenes

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for schema in ["streaming", "upload"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.upload.upload")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.streaming.checkpoints")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

## 3 · Volver al estado inicial

Borra las tablas del laboratorio si existen de una ejecución anterior y vacía los dos volúmenes. Sin esto, una segunda pasada partiría con `001.csv` ya cargado y con el checkpoint de Auto Loader recordando ficheros procesados.

In [ ]:
for tabla in ["sql_csv_autoloader", "python_csv_autoloader"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.streaming.{tabla}")

for volumen in [VOLUMEN_UPLOAD, VOLUMEN_CHECKPOINTS]:
    for item in dbutils.fs.ls(volumen):
        dbutils.fs.rm(item.path, recurse=True)
    print("vaciado", volumen)

## 4 · Cargar `000.csv` y verificar

In [ ]:
import shutil

shutil.copyfile(f"{RUTA_DATASETS}/000.csv", f"{VOLUMEN_UPLOAD}/000.csv")

display(dbutils.fs.ls(VOLUMEN_UPLOAD))

filas = spark.sql(f"""
    SELECT count(*) AS filas
    FROM read_files('{VOLUMEN_UPLOAD}', format => 'csv', sep => '|', header => true)
""").first()["filas"]

assert filas == FILAS_ESPERADAS["000.csv"], f"El volumen tiene {filas} filas, esperaba 3149"
print(f"Entorno listo: {VOLUMEN_UPLOAD} contiene solo 000.csv con {filas} filas.")